# Practical No. 8: Data Processing and Analysis Using PySpark

### **STEP 1 — Install PySpark**

In [ ]:
!pip install pyspark

### **STEP 2 — Import PySpark**

In [ ]:
from pyspark.sql import SparkSession

print("PySpark imported successfully!")

PySpark imported successfully!


### **STEP 3 — Create a Spark Session**

In [ ]:
spark = SparkSession.builder \
    .appName("PySpark_Practical_8") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


In [ ]:
print("Spark Version:", spark.version)

Spark Version: 4.0.3


### **STEP 4 — Create Our Sales CSV Dataset**

In [ ]:
import pandas as pd

sales_data = {
    "Product_ID": [101, 102, 103, 104, 105, 106, 107, 108],
    "Product": [
        "Laptop",
        "Mouse",
        "Chair",
        "Table",
        "Keyboard",
        "Mobile",
        "Desk",
        "Monitor"
    ],
    "Category": [
        "Electronics",
        "Electronics",
        "Furniture",
        "Furniture",
        "Electronics",
        "Electronics",
        "Furniture",
        "Electronics"
    ],
    "Sales": [
        50000,
        2000,
        8000,
        15000,
        3000,
        30000,
        12000,
        18000
    ],
    "Quantity": [
        2,
        5,
        3,
        2,
        4,
        3,
        2,
        2
    ]
}

sales_df = pd.DataFrame(sales_data)

sales_df.to_csv(
    "sales.csv",
    index=False
)

print("sales.csv created successfully!")

sales.csv created successfully!


### **STEP 5 — Read the CSV Using PySpark**

In [ ]:
df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

print("CSV dataset loaded successfully!")

CSV dataset loaded successfully!


### **STEP 6 — Display the Dataset**

In [ ]:
df.show()

+----------+--------+-----------+-----+--------+
|Product_ID| Product|   Category|Sales|Quantity|
+----------+--------+-----------+-----+--------+
|       101|  Laptop|Electronics|50000|       2|
|       102|   Mouse|Electronics| 2000|       5|
|       103|   Chair|  Furniture| 8000|       3|
|       104|   Table|  Furniture|15000|       2|
|       105|Keyboard|Electronics| 3000|       4|
|       106|  Mobile|Electronics|30000|       3|
|       107|    Desk|  Furniture|12000|       2|
|       108| Monitor|Electronics|18000|       2|
+----------+--------+-----------+-----+--------+



### **STEP 7 — Display the Schema**

In [ ]:
df.printSchema()

root
 |-- Product_ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sales: integer (nullable = true)
 |-- Quantity: integer (nullable = true)



### **STEP 8 — Select Specific Columns**

In [ ]:
df.select(
    "Product",
    "Category",
    "Sales"
).show()

+--------+-----------+-----+
| Product|   Category|Sales|
+--------+-----------+-----+
|  Laptop|Electronics|50000|
|   Mouse|Electronics| 2000|
|   Chair|  Furniture| 8000|
|   Table|  Furniture|15000|
|Keyboard|Electronics| 3000|
|  Mobile|Electronics|30000|
|    Desk|  Furniture|12000|
| Monitor|Electronics|18000|
+--------+-----------+-----+



### **STEP 9 — Filtering**

In [ ]:
filtered_df = df.filter(
    df["Sales"] > 10000
)

filtered_df.show()

+----------+-------+-----------+-----+--------+
|Product_ID|Product|   Category|Sales|Quantity|
+----------+-------+-----------+-----+--------+
|       101| Laptop|Electronics|50000|       2|
|       104|  Table|  Furniture|15000|       2|
|       106| Mobile|Electronics|30000|       3|
|       107|   Desk|  Furniture|12000|       2|
|       108|Monitor|Electronics|18000|       2|
+----------+-------+-----------+-----+--------+



### **STEP 10 — Another Filtering Example**

In [ ]:
electronics_df = df.filter(
    df["Category"] == "Electronics"
)

electronics_df.show()

+----------+--------+-----------+-----+--------+
|Product_ID| Product|   Category|Sales|Quantity|
+----------+--------+-----------+-----+--------+
|       101|  Laptop|Electronics|50000|       2|
|       102|   Mouse|Electronics| 2000|       5|
|       105|Keyboard|Electronics| 3000|       4|
|       106|  Mobile|Electronics|30000|       3|
|       108| Monitor|Electronics|18000|       2|
+----------+--------+-----------+-----+--------+



# STEP 11 — Grouping and Aggregation

### **Step 11.1 — Group Sales by Category**

In [ ]:
category_sales = df.groupBy("Category").sum("Sales")

category_sales.show()

+-----------+----------+
|   Category|sum(Sales)|
+-----------+----------+
|Electronics|    103000|
|  Furniture|     35000|
+-----------+----------+



### **STEP 11.2 — Use agg() for Better Aggregation**

In [ ]:
from pyspark.sql.functions import sum

In [ ]:
category_sales = df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales")
)

category_sales.show()

+-----------+-----------+
|   Category|Total_Sales|
+-----------+-----------+
|Electronics|     103000|
|  Furniture|      35000|
+-----------+-----------+



### **STEP 11.3 — Count Products in Each Category**

In [ ]:
from pyspark.sql.functions import count

In [ ]:
product_count = df.groupBy("Category").agg(
    count("Product_ID").alias("Product_Count")
)

product_count.show()

+-----------+-------------+
|   Category|Product_Count|
+-----------+-------------+
|Electronics|            5|
|  Furniture|            3|
+-----------+-------------+



### **STEP 11.4 — Calculate Average Sales**

In [ ]:
from pyspark.sql.functions import avg

In [ ]:
average_sales = df.groupBy("Category").agg(
    avg("Sales").alias("Average_Sales")
)

average_sales.show()

+-----------+------------------+
|   Category|     Average_Sales|
+-----------+------------------+
|Electronics|           20600.0|
|  Furniture|11666.666666666666|
+-----------+------------------+



### **STEP 11.5 — Perform Multiple Aggregations Together**

In [ ]:
summary = df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    count("Product_ID").alias("Product_Count")
)

summary.show()

+-----------+-----------+------------------+-------------+
|   Category|Total_Sales|     Average_Sales|Product_Count|
+-----------+-----------+------------------+-------------+
|Electronics|     103000|           20600.0|            5|
|  Furniture|      35000|11666.666666666666|            3|
+-----------+-----------+------------------+-------------+



### **STEP 11.6 — Sort the Result**

In [ ]:
summary.orderBy(
    "Total_Sales",
    ascending=False
).show()

+-----------+-----------+------------------+-------------+
|   Category|Total_Sales|     Average_Sales|Product_Count|
+-----------+-----------+------------------+-------------+
|Electronics|     103000|           20600.0|            5|
|  Furniture|      35000|11666.666666666666|            3|
+-----------+-----------+------------------+-------------+



# STEP 12 — Remove Duplicate Records

### **Step 12.1 — Create a Dataset with a Duplicate**

In [ ]:
duplicate_row = df.filter(
    df["Product_ID"] == 101
)

df_with_duplicate = df.union(duplicate_row)

print("Dataset with duplicate record:")
df_with_duplicate.show()

Dataset with duplicate record:
+----------+--------+-----------+-----+--------+
|Product_ID| Product|   Category|Sales|Quantity|
+----------+--------+-----------+-----+--------+
|       101|  Laptop|Electronics|50000|       2|
|       102|   Mouse|Electronics| 2000|       5|
|       103|   Chair|  Furniture| 8000|       3|
|       104|   Table|  Furniture|15000|       2|
|       105|Keyboard|Electronics| 3000|       4|
|       106|  Mobile|Electronics|30000|       3|
|       107|    Desk|  Furniture|12000|       2|
|       108| Monitor|Electronics|18000|       2|
|       101|  Laptop|Electronics|50000|       2|
+----------+--------+-----------+-----+--------+



### **Step 12.2 — Count the Records**

In [ ]:
print("Number of records:", df_with_duplicate.count())

Number of records: 9


### **Step 12.3 — Remove Duplicate Records**

In [ ]:
df_no_duplicates = df_with_duplicate.dropDuplicates()

In [ ]:
df_no_duplicates.show()

+----------+--------+-----------+-----+--------+
|Product_ID| Product|   Category|Sales|Quantity|
+----------+--------+-----------+-----+--------+
|       106|  Mobile|Electronics|30000|       3|
|       105|Keyboard|Electronics| 3000|       4|
|       102|   Mouse|Electronics| 2000|       5|
|       101|  Laptop|Electronics|50000|       2|
|       104|   Table|  Furniture|15000|       2|
|       107|    Desk|  Furniture|12000|       2|
|       103|   Chair|  Furniture| 8000|       3|
|       108| Monitor|Electronics|18000|       2|
+----------+--------+-----------+-----+--------+



### **Step 12.4 — Check the Number of Records**

In [ ]:
print(
    "Records after removing duplicates:",
    df_no_duplicates.count()
)

Records after removing duplicates: 8


### **Step 12.5 — Remove Duplicates Based on a Specific Column**

In [ ]:
df_unique = df_with_duplicate.dropDuplicates(
    ["Product_ID"]
)

df_unique.show()

+----------+--------+-----------+-----+--------+
|Product_ID| Product|   Category|Sales|Quantity|
+----------+--------+-----------+-----+--------+
|       101|  Laptop|Electronics|50000|       2|
|       102|   Mouse|Electronics| 2000|       5|
|       103|   Chair|  Furniture| 8000|       3|
|       104|   Table|  Furniture|15000|       2|
|       105|Keyboard|Electronics| 3000|       4|
|       106|  Mobile|Electronics|30000|       3|
|       107|    Desk|  Furniture|12000|       2|
|       108| Monitor|Electronics|18000|       2|
+----------+--------+-----------+-----+--------+



# STEP 13 — Join Two Datasets

### **Step 13.1 — Create the Supplier Dataset**

In [ ]:
supplier_data = [
    (101, "ABC Suppliers"),
    (102, "XYZ Suppliers"),
    (103, "Furniture Hub"),
    (104, "Furniture Hub"),
    (105, "Tech World"),
    (106, "Mobile World"),
    (107, "Furniture Hub"),
    (108, "Display Solutions")
]

supplier_columns = [
    "Product_ID",
    "Supplier"
]

supplier_df = spark.createDataFrame(
    supplier_data,
    supplier_columns
)

print("Supplier dataset created successfully!")

Supplier dataset created successfully!


### **Step 13.2 — Display the Supplier Dataset**

In [ ]:
supplier_df.show()

+----------+-----------------+
|Product_ID|         Supplier|
+----------+-----------------+
|       101|    ABC Suppliers|
|       102|    XYZ Suppliers|
|       103|    Furniture Hub|
|       104|    Furniture Hub|
|       105|       Tech World|
|       106|     Mobile World|
|       107|    Furniture Hub|
|       108|Display Solutions|
+----------+-----------------+



### **STEP 13.3 — Perform Inner Join**

In [ ]:
joined_df = df.join(
    supplier_df,
    on="Product_ID",
    how="inner"
)

joined_df.show()

+----------+--------+-----------+-----+--------+-----------------+
|Product_ID| Product|   Category|Sales|Quantity|         Supplier|
+----------+--------+-----------+-----+--------+-----------------+
|       101|  Laptop|Electronics|50000|       2|    ABC Suppliers|
|       102|   Mouse|Electronics| 2000|       5|    XYZ Suppliers|
|       103|   Chair|  Furniture| 8000|       3|    Furniture Hub|
|       104|   Table|  Furniture|15000|       2|    Furniture Hub|
|       105|Keyboard|Electronics| 3000|       4|       Tech World|
|       106|  Mobile|Electronics|30000|       3|     Mobile World|
|       107|    Desk|  Furniture|12000|       2|    Furniture Hub|
|       108| Monitor|Electronics|18000|       2|Display Solutions|
+----------+--------+-----------+-----+--------+-----------------+



### **STEP 13.4 — Check the Schema of Joined Data**

In [ ]:
joined_df.printSchema()

root
 |-- Product_ID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sales: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Supplier: string (nullable = true)



### **STEP 13.5 — Perform a Left Join**

In [ ]:
left_join_df = df.join(
    supplier_df,
    on="Product_ID",
    how="left"
)

left_join_df.show()

+----------+--------+-----------+-----+--------+-----------------+
|Product_ID| Product|   Category|Sales|Quantity|         Supplier|
+----------+--------+-----------+-----+--------+-----------------+
|       107|    Desk|  Furniture|12000|       2|    Furniture Hub|
|       103|   Chair|  Furniture| 8000|       3|    Furniture Hub|
|       104|   Table|  Furniture|15000|       2|    Furniture Hub|
|       106|  Mobile|Electronics|30000|       3|     Mobile World|
|       105|Keyboard|Electronics| 3000|       4|       Tech World|
|       108| Monitor|Electronics|18000|       2|Display Solutions|
|       101|  Laptop|Electronics|50000|       2|    ABC Suppliers|
|       102|   Mouse|Electronics| 2000|       5|    XYZ Suppliers|
+----------+--------+-----------+-----+--------+-----------------+



### **STEP 13.6 — Perform a Right Join**

In [ ]:
right_join_df = df.join(
    supplier_df,
    on="Product_ID",
    how="right"
)

right_join_df.show()

+----------+--------+-----------+-----+--------+-----------------+
|Product_ID| Product|   Category|Sales|Quantity|         Supplier|
+----------+--------+-----------+-----+--------+-----------------+
|       101|  Laptop|Electronics|50000|       2|    ABC Suppliers|
|       102|   Mouse|Electronics| 2000|       5|    XYZ Suppliers|
|       103|   Chair|  Furniture| 8000|       3|    Furniture Hub|
|       104|   Table|  Furniture|15000|       2|    Furniture Hub|
|       105|Keyboard|Electronics| 3000|       4|       Tech World|
|       106|  Mobile|Electronics|30000|       3|     Mobile World|
|       107|    Desk|  Furniture|12000|       2|    Furniture Hub|
|       108| Monitor|Electronics|18000|       2|Display Solutions|
+----------+--------+-----------+-----+--------+-----------------+



### **STEP 13.7 — Which Join Are We Using for the Practical?**

In [ ]:
joined_df = df.join(
    supplier_df,
    on="Product_ID",
    how="inner"
)

# STEP 14 — Calculate Average Sales by Product Category

In [ ]:
from pyspark.sql.functions import avg

### **Step 14.1 — Calculate Average Sales**

In [ ]:
average_sales = df.groupBy("Category").agg(
    avg("Sales").alias("Average_Sales")
)

average_sales.show()

+-----------+------------------+
|   Category|     Average_Sales|
+-----------+------------------+
|Electronics|           20600.0|
|  Furniture|11666.666666666666|
+-----------+------------------+



### **Step 14.2 — Round the Average**

In [ ]:
from pyspark.sql.functions import round

In [ ]:
average_sales = df.groupBy("Category").agg(
    round(
        avg("Sales"),
        2
    ).alias("Average_Sales")
)

average_sales.show()

+-----------+-------------+
|   Category|Average_Sales|
+-----------+-------------+
|Electronics|      20600.0|
|  Furniture|     11666.67|
+-----------+-------------+



### **Step 14.4 — Sort by Average Sales**

In [ ]:
average_sales.orderBy(
    "Average_Sales",
    ascending=False
).show()

+-----------+-------------+
|   Category|Average_Sales|
+-----------+-------------+
|Electronics|      20600.0|
|  Furniture|     11666.67|
+-----------+-------------+



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, count

# Create Spark session
spark = SparkSession.builder \
    .appName("Practical_8_PySpark") \
    .getOrCreate()

# Create sample datasets
sales_data = [
    (1, "Laptop", "Electronics", 55000),
    (2, "Mobile", "Electronics", 25000),
    (3, "Shirt", "Clothing", 1500),
    (4, "Jeans", "Clothing", 2500),
    (5, "Tablet", "Electronics", 18000),
    (6, "Shoes", "Footwear", 3000),
    (7, "Headphones", "Electronics", 2000),
    (8, "Jacket", "Clothing", 3500),
    (9, "Watch", "Accessories", 4000),
    (10, "Bag", "Accessories", 2200),
    (3, "Shirt", "Clothing", 1500)
]

sales_columns = ["SaleID", "Product", "Category", "Sales"]

sales_df = spark.createDataFrame(sales_data, sales_columns)

customer_data = [
    (1, "Customer A"),
    (2, "Customer B"),
    (3, "Customer C"),
    (4, "Customer D"),
    (5, "Customer E"),
    (6, "Customer F"),
    (7, "Customer G"),
    (8, "Customer H"),
    (9, "Customer I"),
    (10, "Customer J")
]

customer_columns = ["SaleID", "CustomerName"]

customers_df = spark.createDataFrame(customer_data, customer_columns)

# Write and read CSV dataset
csv_path = "sales_data.csv"
sales_df.write.mode("overwrite").option("header", True).csv(csv_path)

df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

print("CSV Dataset:")
df.show()

# Display schema
print("Schema:")
df.printSchema()

# Filtering
print("Filtered Sales above 5000:")
df.filter(col("Sales") > 5000).show()

# Grouping and aggregation
print("Total Sales by Category:")
df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales"),
    count("SaleID").alias("Number_of_Sales")
).show()

# Remove duplicate records
df_unique = df.dropDuplicates()

print("Dataset after removing duplicates:")
df_unique.show()

# Join two datasets
joined_df = df_unique.join(customers_df, on="SaleID", how="inner")

print("Joined Dataset:")
joined_df.show()

# Average sales by product category
print("Average Sales by Product Category:")
average_sales = df_unique.groupBy("Category").agg(
    avg("Sales").alias("Average_Sales")
)

average_sales.show()

# Stop Spark session
spark.stop()

CSV Dataset:
+------+----------+-----------+-----+
|SaleID|   Product|   Category|Sales|
+------+----------+-----------+-----+
|     6|     Shoes|   Footwear| 3000|
|     7|Headphones|Electronics| 2000|
|     8|    Jacket|   Clothing| 3500|
|     9|     Watch|Accessories| 4000|
|    10|       Bag|Accessories| 2200|
|     3|     Shirt|   Clothing| 1500|
|     1|    Laptop|Electronics|55000|
|     2|    Mobile|Electronics|25000|
|     3|     Shirt|   Clothing| 1500|
|     4|     Jeans|   Clothing| 2500|
|     5|    Tablet|Electronics|18000|
+------+----------+-----------+-----+

Schema:
root
 |-- SaleID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sales: integer (nullable = true)

Filtered Sales above 5000:
+------+-------+-----------+-----+
|SaleID|Product|   Category|Sales|
+------+-------+-----------+-----+
|     1| Laptop|Electronics|55000|
|     2| Mobile|Electronics|25000|
|     5| Tablet|Electronics|18000|
+------+-

In [2]:
print("Name: Mayuri Narsu Patil")

Name: Mayuri Narsu Patil
